# Homework 02: SQL Mini-Project with Sakila
## Joins, Aggregation, CTEs, Window Functions, and Performance

**Due date:** _April 3, 2026 by 11:59 PM_

**Points:** 100 total

KAYLA KRAMER 

- - -

## Overview and Instructions

**Context:** This homework builds directly on our relational database + SQL module (Days 11–19). You will use a Dockerized MySQL database (the Sakila sample database you set up previously) and a Jupyter notebook to carry out a small, portfolio-ready SQL analysis.

You will:
- Connect to a Dockerized MySQL database from this notebook.
- Design and implement several non-trivial SQL queries using Sakila.
- Use joins, grouped aggregation with `HAVING`, CTEs, window functions, and a brief performance check.
- Practice explicit validation habits (row counts, sanity checks, cross-checks).
- Produce at least one manager-style report table that could stand alone in a portfolio.
- Reflect on your SQL reasoning and debugging process.
- Log any use of AI tools in an **AI Audit Log** section.
- Create a GitHub repository to share your work.

**If you get stuck, have questions, or run into any issues as you work through this assignment**, remember to consult the PCAs, ICAs, slides from class for the **Day 11-19 class periods** -- the slides in particular often have worked examples of queries that are similar to what you'll need to write for this assignment. You are also encouraged to **come to office hours** to get direct support from your instructors! If you can't find out office hours information or need to schedule by appoint, **send us an email or catch one of us at the end of class**.

**Submission:**
- Push this completed notebook (and any small helper files) to a new private GitHub repo called `cmse492-hw02-yourMSUNetID` (make sure you use your actual MSU NetID!). **Details for this process are include in Part 10 below.**
- Submit the GitHub URL via the provided Microsoft form.
- Upload the executed notebook (with outputs) to D2L as a backup.

This notebook is designed to be a **self-contained artifact**: someone with access to your Docker image and credentials should be able to rerun your analysis and understand what you did, why you did it, and how you validated it.

**Grading breakdown:** See section headings for point values.

<div class="alert alert-attention">

**Note about AI tools:** As discussed at the beginning of the course, you are allowed to use AI tools (e.g., ChatGPT, Copilot) in responsible, transparent, and ethical ways. For this particular assignment, if you end up using AI tools for assignment, but you will be expected to document your usage in an "AI Audit Log" section near the end of this notebook. See the details in the "AI Audit Log" section below.

</div>


- - -

## 0. Setup and Database Connection

In this section you will:
- Confirm your Dockerized MySQL + Sakila setup is working.
- Establish a connection from Python to the database using `sqlalchemy`.
- Define a small helper function to run SQL and get results as a `pandas` DataFrame (make sure you look at this function so that you understand how it works and can use it in the assignment!)

**Even though you are being provided with the code cell necessary to connect to the database, you should make sure you carefully review the code and understand how it is working.** If you end up needing to connect to a SQL database on your own in the future, you should know how to do so.

**Remember**: Before you try to connect to the database, make sure your Docker container is running and that the Sakila database is properly set up inside it. If you don't recall how to do this, review the instructions from the Day 17 content and get that set up first.

If you run into an issue with this part of the assignment, contact your instructors _as soon as possible_ so we can help you get it resolved. You will not be able to complete the rest of the assignment without a working database connection, so this is a critical first step.


In [170]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Credentials for the local Docker MySQL container (must match docker-compose.yml)
uname = 'sakila'
pwd = 'p_ssW0rd'
hname = 'localhost'
dbname = 'sakila'

engine = create_engine(f'mysql+pymysql://{uname}:{pwd}@{hname}/{dbname}')
connection = engine.connect()

def run_sql(query, params=None):
    """Run a SQL query and return a pandas DataFrame.
    `query` can be a string; `params` is an optional dict for parameterized queries.
    """
    with engine.connect() as conn:
        result = conn.execute(text(query), params or {})
        df = pd.DataFrame(result.fetchall(), columns=result.keys())
    return df

# Quick test to confirm that you can connect and query the database
test_df = run_sql("SELECT * FROM film LIMIT 5;")
test_df.head()

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2006-02-15 05:03:42
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,1,None,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2006-02-15 05:03:42
3,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,1,None,5,2.99,117,26.99,G,"Commentaries,Behind the Scenes",2006-02-15 05:03:42
4,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,1,None,6,2.99,130,22.99,G,Deleted Scenes,2006-02-15 05:03:42


In [172]:
def run_sql(query, params=None):
    with engine.connect() as conn:
        result = conn.execute(text(query), params or {})
        df = pd.DataFrame(result.fetchall(), columns=result.keys())
    return df 
#USED FROM CHATGPT BECAUSE THIS CODE HELPED MY JUPITER NOTEBOOK CONNECT TO SALIKA DATABASE

- - -

## 1. Framing Your Mini-Project (5 points)

Your goal is to design a **small analytic story** using the Sakila database. Think of a manager at a video rental company who wants to understand something about customers, films, inventory, stores, or revenue.

In this section, you will define:
- A brief business/analytic question or set of questions you want to answer using SQL and the audience you are imagining.
- The main tables you expect to use.
- The kinds of outputs you want (e.g., a ranked table, grouped summary, percent-of-total view).

**Note**: you might find that your question and plans evolve as you start writing SQL and seeing results. This is totally normal! Just make sure to document your initial thinking here, and then as you evolve your question and plans, make sure to update this section to reflect that evolution. Make sure to keep your initial question and plans in there as well, so that we can see how your thinking evolved.

**Another note**: You might need to spend some time exploring the Sakila schema and data to come up with a question that is interesting and feasible. You can access the DB using Adminer (via `http://localhost:8080`) to explore the schema and/or use SQL queries to poke at the data as part of your process of coming up with a good question.

If you're really struggling to come up with your "story", you can start working through the SQL exercises in the sections that follow, which require you to come up with individual questions to answer with SQL. **You might find that as you do those exercises, you see a collection of insights that your can develop into your project story.** "Reverse-engineering" your project story from the SQL exercises is a totally valid way to come up with a project question and plan!

### 1.1 Project question and plan

**Prompt:** In 1–2 paragraphs, describe your mini-project:
- What is the main question or set of questions you want to answer using SQL?
- Who is the imagined audience (e.g., store manager, marketing analyst, operations lead)?
- Which core tables do you expect to rely on (list at least 3 Sakila tables and why)?
- What kind of final table(s) or figure(s) do you expect to produce (e.g., top categories by revenue, customer segments by activity)?


My project  focuses on understanding customers, film categories, and store locations generate the most rental activity and revenue in the Sakila database. The main questions I want to answer are which categories are rented the most, which customers spend the most money, and how store-level performance differs based on rentals and payments. I chose this topic because it connects customer behavior, film demand, and business performance in a way that would be useful for decision-making.

The imagined audience for this project is a store manager or operations analyst at a video rental company who wants to better understand customer rental behavior and revenue patterns. The core tables I expect to rely on are customer, rental, payment, inventory, film, film_category, category, and store. These tables help connect customers, transactions, films, categories, and store locations. My final outputs will include a ranked table of top customers by spending, a grouped summary of top film categories by rentals, and a manager-style report comparing store performance.

---

## 2. Joins and Multi-Table Queries (15 points)

In this section, you will write **at least two non-trivial join queries** that connect 2–3 tables with meaningful conditions (not just a single key lookup). For each query:
- Clearly state the question it answers.
- Sketch what a single row in the result represents.
- Write and run the SQL.
- Perform at least one validation check (row counts, spot-check rows against base tables, etc.).


### 2.1 Join Query 1

**Example idea (you may adapt this or pick a different one, do not use exactly this example):**
- "For each rental in a specified 60-day period, show customer name, film title, rental date, and store city."

Based on your question and plans from the previous section, write your own question and design for this first join query. You can use the example above as a template or inspiration, but make sure to write your own unique question and design that fits with your overall mini-project. For this first join query, you're expected to clearly define the following:

1. **Question in words**: Describe what you are trying to learn in one or two sentences.
2. **Row meaning sketch**: What does each row represent? Which columns should appear?
3. **Tables and keys**: Which tables are you joining (e.g., `rental`, `customer`, `inventory`, `film`, `store`, `address`, `city`)? On which key columns?
4. **SQL query**: Write the join with clear aliases.
5. **Validation**: Perform at least one validation check, such as:
   - Compare a row count to a simpler query.
   - Spot-check an individual record across base tables.


> **2.1.a Question and design**

Which customers rented certain films?
Each row represents one customer summary rather than one individual rental.

The main tables used are customer, rental, and payment. The keys used are customer.customer_id = rental.customer_id and customer.customer_id = payment.customer_id.

The output includes the customer ID, customer name, total number of rentals, and total amount spent.

In [180]:
join_query1 = """
SELECT 
    r.rental_id,
    c.first_name,
    c.last_name,
    f.title AS film_title,
    r.rental_date,
    i.store_id
FROM rental AS r
JOIN customer AS c
    ON r.customer_id = c.customer_id
JOIN inventory AS i
    ON r.inventory_id = i.inventory_id
JOIN film AS f
    ON i.film_id = f.film_id
ORDER BY r.rental_date DESC
LIMIT 20;
"""

data1join = run_sql(join_query1)
data1join.head()

,rental_id,first_name,last_name,film_title,rental_date,store_id
0,11739,LOUIS,LEONE,ZHIVAGO CORE,2006-02-14 15:16:03,2
1,14616,NEIL,RENNER,WORLD LEATHERNECKS,2006-02-14 15:16:03,2
2,11676,NATALIE,MEYER,WOMEN DORADO,2006-02-14 15:16:03,1
3,15966,JEREMY,HURTADO,WINDOW SIDE,2006-02-14 15:16:03,2
4,13486,NAOMI,JENNINGS,WILD APOLLO,2006-02-14 15:16:03,1


**2.1.c Validation for Join Query 1**

Describe at least one validation check you ran and what you concluded based on it.

You might:
- Show a supporting SQL snippet in a small code cell (e.g., `COUNT(*)` comparison).
- Spot-check one ID across multiple tables and describe what you saw.


In [183]:
val_query1 = """
SELECT COUNT(*) AS total_rentals
FROM rental;
"""

run_sql(val_query1)

,total_rentals
0,16044


I ran a COUNT(*) query on the rental table and confirmed that there are 16,044 rental transactions in total. Since each row in my join query is intended to represent one rental transaction, this count gave me a useful reference point for checking whether the join structure made sense. I also spot-checked the customer names, film titles, rental dates, and store IDs in the sample output to make sure the joined columns appeared consistent with the expected rental records.

### 2.2 Join Query 2

Create a **different** join that still aligns with your project theme.

**Example ideas (again, you should adapt these to your project or create your own):**
- Customer-level summary: each row is one customer with their total rentals, number of distinct categories they rent, and home city.
- Inventory-level view: each row is one film copy with film title, store, and how many times it has been rented.

Repeat the pattern as before, clearly defining:
1. Question in words.
2. Row meaning sketch.
3. Tables and keys.
4. SQL query.
5. Validation check(s).

> **2.2.a Question and design**
> 
QUESTION: Which film categories have the highest rental activity? Which categories have at least 500 rentals? 
This can help the store manager understand which types of films are most popular to determine inventory amount.


In [188]:
join_query2 = """
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    rental_summary.total_rentals,
    payment_summary.total_spent
FROM customer AS c
JOIN (
    SELECT
        customer_id,
        COUNT(*) AS total_rentals
    FROM rental
    GROUP BY customer_id
) AS rental_summary
    ON c.customer_id = rental_summary.customer_id
JOIN (
    SELECT
        customer_id,
        ROUND(SUM(amount), 2) AS total_spent
    FROM payment
    GROUP BY customer_id
) AS payment_summary
    ON c.customer_id = payment_summary.customer_id
ORDER BY payment_summary.total_spent DESC
LIMIT 20;
"""
join_2 = run_sql(join_query2)
join_2.head()

,customer_id,first_name,last_name,total_rentals,total_spent
0,526,KARL,SEAL,45,221.55
1,148,ELEANOR,HUNT,46,216.54
2,144,CLARA,SHAW,42,195.58
3,178,MARION,SNYDER,39,194.61
4,137,RHONDA,KENNEDY,39,194.61


**2.2.c Validation for Join Query 2**

As before, describe your validation steps and conclusions for this second join query. You can use similar validation techniques as before or come up with new ones that fit the specific question and data you are working with.

> **_Describe your validation steps and conclusions here._**

---

## 3. Grouped Aggregation with HAVING (15 points)

In this section you will:
- Write at least one grouped aggregation that uses `GROUP BY` and `HAVING`.
- Apply sanity checks (e.g., sum of group counts vs. raw counts, spot-check of a group).
- Tie the result to a question your audience might actually care about.


### 3.1 Aggregation Query with HAVING (required)

**Example ideas (adapt or design your own to fit your audience and align with your mini-project):**
- "Find film categories with at least 1000 total rentals, ordered from most to least rented."
- "Find customers who have spent at least \$X in total payments."

For your query, **document**:
1. Business question in words (1–2 sentences).
2. Row meaning and expected columns in the grouped result.
3. SQL query with `GROUP BY` and `HAVING`.
4. At least two sanity checks (e.g., compare grouped count sums to raw counts, spot-check a single group).


> **3.1.a Question and design**
> 

Which film categories have the highest rental activity?
Which categories have at least 500 rentals?
The row in the result represents one film category. The output includes the category name and the total number of rentals associated with films in that category. I expect the resulting table to omly show only categories whose rental count passes the threshold in the HAVING clause.

In [195]:
agg_query = """
SELECT
    c.name AS category_name,
    COUNT(*) AS total_rentals
FROM rental AS r
JOIN inventory AS i
    ON r.inventory_id = i.inventory_id
JOIN film AS f
    ON i.film_id = f.film_id
JOIN film_category AS fc
    ON f.film_id = fc.film_id
JOIN category AS c
    ON fc.category_id = c.category_id
GROUP BY c.name
HAVING COUNT(*) >= 500
ORDER BY total_rentals DESC;
"""

agg = run_sql(agg_query)
agg

,category_name,total_rentals
0,Sports,1179
1,Animation,1166
2,Action,1112
3,Sci-Fi,1101
4,Family,1096
5,Drama,1060
6,Documentary,1050
7,Foreign,1033
8,Games,969
9,Children,945


**3.1.c Sanity checks for aggregation**

Describe at least **two** checks you performed. Ideas:
- Compare the sum of group counts to a raw `COUNT(*)` over a comparable subset.
- Focus on one group (e.g., one category or one customer), query its underlying rows, and verify the count.
- If applicable, use a `LEFT JOIN` to see categories with zero values and explain what you observe.


In [198]:
CHECK1= """
SELECT
    c.name AS category_name,
    COUNT(*) AS total_rentals
FROM rental AS r
JOIN inventory AS i
    ON r.inventory_id = i.inventory_id
JOIN film AS f
    ON i.film_id = f.film_id
JOIN film_category AS fc
    ON f.film_id = fc.film_id
JOIN category AS c
    ON fc.category_id = c.category_id
GROUP BY c.name
ORDER BY total_rentals DESC;
"""

run_sql(CHECK1)

,category_name,total_rentals
0,Sports,1179
1,Animation,1166
2,Action,1112
3,Sci-Fi,1101
4,Family,1096
5,Drama,1060
6,Documentary,1050
7,Foreign,1033
8,Games,969
9,Children,945


In [200]:
check2 = """
SELECT
    c.name AS category_name,
    COUNT(*) AS horror_total_rentals
FROM rental AS r
JOIN inventory AS i
    ON r.inventory_id = i.inventory_id
JOIN film AS f
    ON i.film_id = f.film_id
JOIN film_category AS fc
    ON f.film_id = fc.film_id
JOIN category AS c
    ON fc.category_id = c.category_id
WHERE c.name = 'Horror'
GROUP BY c.name;
"""

run_sql(check2)

,category_name,horror_total_rentals
0,Horror,846


I performed two sanity checks for this aggregation query. First, I ran the same grouped query without the HAVING clause so I could compare the full list of categories to the filtered result. This confirmed that my final output was correctly showing only the categories with at least 500 rentals and the same specific total rentals amount. Second, I spot-checked one specifc category(I choose Horror), by running a separate count query on the underlying rental rows for that category. This returned 846 total rentals, which exactly matched the value shown in my aggregated result. Thia confirms that the grouped count is being built from real rental rows within that category. These checks gave me confidence that the grouping and filtering logic were working as intended.

- - - 

## 4. Multi-Step Query Using a CTE (15 points)

Now you will use a **CTE (`WITH` clause)** to break a complex analysis into readable steps. You may reuse logic from earlier sections, but the CTE should add structure (e.g., pre-filtering, intermediate grouping) rather than just rename an existing query.

**Example patterns:**
- First CTE: compute customer-level rental counts and total payments.
- Second step: filter to active customers, compute ranks or thresholds on top.

Your CTE query must:
- Use at least one `WITH` clause.
- Do something non-trivial in the CTE (filtering, joining, grouping, etc.).
- Be accompanied by validation (e.g., run parts of the CTE separately, check a subset).

### 4.1 CTE design and explanation

Describe your CTE in words:
- What does the CTE compute?
- What does the outer query do on top of it?
- How does this decomposition make the logic clearer than a single big query would be?


My CTE produces one summarized row per customer, including total rentals and total payments. The CTE creates one row per customer with their customer ID, name, total number of rentals, and total amount spent. This lets me build a clean intermediate table before doing any final filtering or ranking. 

The outer query then uses that customer summary to show the customers with the highest spending, while also restricting the result to customers who have made at least 20 rentals. Breaking the logic into a CTE makes the query easier to read and validate because I can first think about building the customer summary, then separately think about how I want to filter and sort the final result.

### 4.2 Writing the CTE query

Now that you have a plan for your CTE, write the SQL query that implements it.

In [207]:
cte_query = """
WITH rental_summary AS (
    SELECT
        customer_id,
        COUNT(*) AS total_rentals
    FROM rental
    GROUP BY customer_id
),
payment_summary AS (
    SELECT
        customer_id,
        ROUND(SUM(amount), 2) AS total_spent
    FROM payment
    GROUP BY customer_id
),
customer_summary AS (
    SELECT
        c.customer_id,
        c.first_name,
        c.last_name,
        rs.total_rentals,
        ps.total_spent
    FROM customer AS c
    JOIN rental_summary AS rs
        ON c.customer_id = rs.customer_id
    JOIN payment_summary AS ps
        ON c.customer_id = ps.customer_id
)
SELECT
    customer_id,
    first_name,
    last_name,
    total_rentals,
    total_spent
FROM customer_summary
WHERE total_rentals >= 20
ORDER BY total_spent DESC
LIMIT 20;
"""

cte = run_sql(cte_query)
cte.head()

,customer_id,first_name,last_name,total_rentals,total_spent
0,526,KARL,SEAL,45,221.55
1,148,ELEANOR,HUNT,46,216.54
2,144,CLARA,SHAW,42,195.58
3,178,MARION,SNYDER,39,194.61
4,137,RHONDA,KENNEDY,39,194.61


### 4.3 CTE validation

Run at least one validation step focused on the **intermediate CTE logic**, such as:
- Select a subset of the CTE output (e.g., `LIMIT 10`) and manually compare to base tables.
- Recompute a simpler version of a metric (e.g., total payments) for one customer and confirm it matches the CTE result.

In [210]:
check_cte1 = """
SELECT
    customer_id,
    COUNT(*) AS total_rentals
FROM rental
WHERE customer_id = 148
GROUP BY customer_id;
"""

run_sql(check_cte1)

,customer_id,total_rentals
0,148,46


In [212]:
check_cte2 = """
SELECT
    customer_id,
    ROUND(SUM(amount), 2) AS total_spent
FROM payment
WHERE customer_id = 148
GROUP BY customer_id;
"""
run_sql(check_cte2)


,customer_id,total_spent
0,148,216.54


To validate the CTE logic, I selected one customer from the result and recomputed that customer’s rental count and total payment directly from the base rental and payment tables. I then compared those values to the customer’s row in the CTE result. This helped confirm that the intermediate summaries were being calculated correctly before the final filtering and sorting were applied. Because I separated rental counts and payment totals into different CTE steps, the logic was easier to check and less likely to create duplicate-counting problems.



---

## 5. Window Function with `OVER (...)` (15 points)

Next, you will write at least one query that uses a **window function**. Options include:
- `ROW_NUMBER()`, `RANK()` or `DENSE_RANK()` within a category (e.g., top customers per store).
- A percent-of-total calculation using `SUM(...) OVER ()` or `SUM(...) OVER (PARTITION BY ...)`.

Your window query should:
- Use `OVER (...)` in a meaningful way.
- Produce a result that could help your audience understand ranking or relative importance.
- Include at least one validation step (e.g., verify that percentages sum to ~100%).


### 5.1 Window function design

Describe your plan:
- What are you ranking or computing percentages over?
- What does each row represent in the final result?
- How will your audience interpret the window column(s)?


For the window function section, I want to calculate how much each film category contributes to total rental activity across the database. Each row in the final result will represent one film category, along with its total rentals and its percentage share of all rentals. This helps show not just which categories are popular, but also how important each category is relative to the overall business.

My audience would interpret this result as a way to compare category performance beyond raw counts alone. The window function is useful here because it allows me to divide each category’s rental total by the grand total across all categories without collapsing the grouped result into a separate summary query.

### 5.2 Writing the window function query

Now that you have a plan for your window function, write the SQL query that implements it.

In [219]:
window_query = """
WITH category_rentals AS (
    SELECT
        c.name AS category_name,
        COUNT(*) AS total_rentals
    FROM rental AS r
    JOIN inventory AS i
        ON r.inventory_id = i.inventory_id
    JOIN film AS f
        ON i.film_id = f.film_id
    JOIN film_category AS fc
        ON f.film_id = fc.film_id
    JOIN category AS c
        ON fc.category_id = c.category_id
    GROUP BY c.name
)
SELECT
    category_name,
    total_rentals,
    ROUND(100.0 * total_rentals / SUM(total_rentals) OVER (), 2) AS pct_of_total_rentals
FROM category_rentals
ORDER BY total_rentals DESC;
"""

window = run_sql(window_query)
window

,category_name,total_rentals,pct_of_total_rentals
0,Sports,1179,7.35
1,Animation,1166,7.27
2,Action,1112,6.93
3,Sci-Fi,1101,6.86
4,Family,1096,6.83
5,Drama,1060,6.61
6,Documentary,1050,6.54
7,Foreign,1033,6.44
8,Games,969,6.04
9,Children,945,5.89


### 5.3 Window function validation

Identify at least one validation you can perform, such as:
- Summing `pct_of_total` and confirming it is close to 100 (allowing for rounding).
- Checking that the ranking order matches the sorted totals.
- Comparing one row’s percent-of-total to a hand calculation.

In [222]:
check_window= """
SELECT
    c.name AS category_name,
    COUNT(*) AS total_rentals
FROM rental AS r
JOIN inventory AS i
    ON r.inventory_id = i.inventory_id
JOIN film AS f
    ON i.film_id = f.film_id
JOIN film_category AS fc
    ON f.film_id = fc.film_id
JOIN category AS c
    ON fc.category_id = c.category_id
WHERE c.name = 'Sports'
GROUP BY c.name;
"""

run_sql(check_window)

,category_name,total_rentals
0,Sports,1179


To validate this window function query, I selected one category, Sports, and separately calculated its total number of rentals using a grouped count query. I then compared that count to the overall total rental count across all categories and manually computed the percentage. The hand-calculated percentage matched the pct_of_total_rentals value produced by the window function, which confirmed that the SUM(...) OVER () calculation was working correctly.

---

## 6. Performance Check with `EXPLAIN` and/or Index (5 points)

Pick **one** of your more complex queries from the earlier sections (join, aggregation, or window-based) and perform a brief performance analysis using MySQL's `EXPLAIN` statement.

You should:
- State which query you are examining and what aspect of its execution plan you want to inspect.
- Run `EXPLAIN` on the query and capture the output in a DataFrame.
- Identify at least one aspect of the plan (e.g., table scan vs. index usage, join order, estimated rows) and explain why it seems reasonable or concerning.
- Optionally, propose or implement a simple index and compare the `EXPLAIN` output before vs. after.

You do **not** need to time queries precisely or chase microseconds. The focus is on reading and interpreting the query plan.

**Note**: You might not see any "red flags" in the `EXPLAIN` output, especially on a small dataset like Sakila. That's totally fine! The point is to practice reading the plan and thinking about what it means.

### 6.1 Choose a query and plan your analysis

Before running `EXPLAIN`, take a moment to describe your plan:
1. Which query from an earlier section are you examining (e.g., "Section 3 aggregation by category", "Section 5 window query")?
2. Why did you choose this query (e.g., it joins many tables, it aggregates a large dataset)?
3. What do you expect to see in the `EXPLAIN` output (e.g., index usage on join keys, a full scan on a particular table)?

For the performance check, I chose the aggregation query from Section 3 that groups rentals by film category and filters categories using HAVING. I chose this query because it joins several tables and performs grouped aggregation, which makes it a good example for inspecting how MySQL executes a more complex analytical query.

In the EXPLAIN output, I expect to see MySQL using indexes on some join keys, while possibly scanning more rows in tables like rental or inventory because those tables are central to the aggregation. I also expect the grouped nature of the query to require extra work compared to a simple lookup query.




### 6.2 Run EXPLAIN on your chosen query

Now that you have a plan, run `EXPLAIN` on the query you chose. Note that you can prefix any SQL query with `EXPLAIN` to get the execution plan — `run_sql()` works with `EXPLAIN` statements just like any other query.

In [229]:
explain_query = """
EXPLAIN
SELECT
    c.name AS category_name,
    COUNT(*) AS total_rentals
FROM rental AS r
JOIN inventory AS i
    ON r.inventory_id = i.inventory_id
JOIN film AS f
    ON i.film_id = f.film_id
JOIN film_category AS fc
    ON f.film_id = fc.film_id
JOIN category AS c
    ON fc.category_id = c.category_id
GROUP BY c.name
HAVING COUNT(*) >= 500
ORDER BY total_rentals DESC;
"""
explain = run_sql(explain_query)
explain

,id,select_type,table,partitions,type,possible_keys,key,key_len,ref,rows,filtered,Extra
0,1,SIMPLE,r,None,index,idx_fk_inventory_id,idx_fk_inventory_id,3,None,1,100.0,Using index; Using temporary; Using filesort
1,1,SIMPLE,i,None,eq_ref,"PRIMARY,idx_fk_film_id",PRIMARY,3,sakila.r.inventory_id,1,100.0,None
2,1,SIMPLE,f,None,eq_ref,PRIMARY,PRIMARY,2,sakila.i.film_id,1,100.0,Using index
3,1,SIMPLE,fc,None,ref,"PRIMARY,fk_film_category_category",PRIMARY,2,sakila.i.film_id,1,100.0,Using index
4,1,SIMPLE,c,None,eq_ref,PRIMARY,PRIMARY,1,sakila.fc.category_id,1,100.0,None


### 6.3 Interpret the EXPLAIN output

Now that you have the `EXPLAIN` output, interpret it in a few sentences. Address at least one of the following:
- Is MySQL using an index or doing a full table scan on any of the tables? Does this seem reasonable given the query structure?
- What is the estimated number of rows being scanned for each table (the `rows` column)? Does this seem appropriate?
- Is the join order what you would expect? Why or why not?
- Based on the plan, would you suggest any changes to improve performance (e.g., adding an index, restructuring the query)?

In [232]:
# Optional: use this cell for any helper queries or pandas checks that
# support your interpretation of the EXPLAIN output.
# For example, you could check the actual row count of a table where EXPLAIN
# shows a large row estimate.

The EXPLAIN output shows that MySQL is using indexes efficiently for most of the joins in this query. The rental table uses the idx_fk_inventory_id index, while the joins to inventory, film, film_category, and category primarily use PRIMARY keys or other referenced keys with eq_ref and ref join types. This is a good sign because it means MySQL is not performing unnecessary full table scans across the joined tables.

One important detail in the Extra column is that the rental table shows Using temporary; Using filesort. This is reasonable because the query performs a GROUP BY and then orders the aggregated result by total rentals, which often requires MySQL to build a temporary structure and sort the grouped output.This behavior is expected since this is an analytical summary query rather than a simple lookup.

The estimated row counts are low across the joined tables, which is appropriate for the Sakila sample database. Overall, the plan suggests that the joins are efficient and that most of the extra work comes from the grouping and sorting logic rather than poor join performance.

### 6.4 Optional: Propose or add an index

If you'd like to go further and it seems like an optimization is needed, you can propose or create a simple index and compare the `EXPLAIN` output before vs. after. Document:
- Which column(s) you chose and why (e.g., frequently used in `WHERE` or join conditions).
- Whether the `EXPLAIN` output changed in a way that suggests better performance (e.g., less full-table scanning, lower estimated rows).

In [236]:
# Optional: create an index and re-run EXPLAIN to compare.
# create_index_sql = """
# -- TODO: Replace with your CREATE INDEX statement.
# -- Example: CREATE INDEX idx_rental_date ON rental(rental_date);
# """
# run_sql(create_index_sql)

# explain_after_index = """
# -- TODO: Replace with your EXPLAIN query after adding the index.
# """
# run_sql(explain_after_index)

---

## 7. Manager-Style Report Table (10 points)

Design **one table** that you would feel comfortable showing to your imagined audience or including in a portfolio. This table should:
- Be relatively **wide** (e.g., 4–8 columns) with clear, human-readable column labels.
- Summarize something decision-relevant (e.g., categories by revenue and share, top customers per store).
- Optionally use `CASE` expressions or window functions to create computed or pivot-like columns.

You may reuse or adapt one of your earlier queries if it already reads like a manager report, or you may write a dedicated query here.

### 7.1 Describe your report

Before writing the SQL, describe your report:
1. Who is the audience for this table (e.g., store manager, marketing analyst, operations lead)?
2. What decision or question does this table support?
3. Which columns will appear, and what does each one tell the audience?
4. Will you reuse a query from an earlier section or write a new one?

This report is designed for a store manager or operations analyst who wants to understand which customers are generating the most revenue and rental activity at each store. The goal of the table is to support decisions related to customer retention, promotions, and store-level performance.

The report includes Store ID, Customer Name, Total Rentals, Total Spent, and Average Spend per Rental. These columns display which customers are most valuable, how active they are, and how customer value compares across stores. I reused and adapted earlier summary logic to build a more presentation-ready report. Also could be repeat in names because customer could vist both stores.


### 7.2 Write the report query

Now write the SQL query that produces your report table. Aim for clear, human-readable column aliases and a layout that could stand on its own in a portfolio or presentation.

In [242]:
report_query = """
WITH rental_summary AS (
    SELECT
        r.customer_id,
        i.store_id,
        COUNT(*) AS total_rentals
    FROM rental AS r
    JOIN inventory AS i
        ON r.inventory_id = i.inventory_id
    GROUP BY r.customer_id, i.store_id
),
payment_summary AS (
    SELECT
        customer_id,
        ROUND(SUM(amount), 2) AS total_spent
    FROM payment
    GROUP BY customer_id
)
SELECT
    rs.store_id AS `Store ID`,
    CONCAT(c.first_name, ' ', c.last_name) AS `Customer Name`,
    rs.total_rentals AS `Total Rentals`,
    ps.total_spent AS `Total Spent`,
    ROUND(ps.total_spent / rs.total_rentals, 2) AS `Avg Spend per Rental`
FROM rental_summary AS rs
JOIN customer AS c
    ON rs.customer_id = c.customer_id
JOIN payment_summary AS ps
    ON rs.customer_id = ps.customer_id
ORDER BY `Total Spent` DESC
LIMIT 20;
"""

report = run_sql(report_query)
report

,Store ID,Customer Name,Total Rentals,Total Spent,Avg Spend per Rental
0,1,KARL SEAL,21,221.55,10.55
1,2,KARL SEAL,24,221.55,9.23
2,1,ELEANOR HUNT,21,216.54,10.31
3,2,ELEANOR HUNT,25,216.54,8.66
4,2,CLARA SHAW,25,195.58,7.82
5,1,CLARA SHAW,17,195.58,11.50
6,2,RHONDA KENNEDY,16,194.61,12.16
7,1,MARION SNYDER,21,194.61,9.27
8,2,MARION SNYDER,18,194.61,10.81
9,1,RHONDA KENNEDY,23,194.61,8.46


### 7.3 Optional: polish in pandas

If you wish, you can use `pandas` to make the report output more presentation-ready:
- Rename columns for readability.
- Format numeric columns (e.g., round percentages, add currency formatting).
- Select or reorder columns for clarity.

This is optional but can help the output feel more portfolio-ready.

In [245]:
# Optional: light pandas polishing for the report table, then display it.
# For example, you could format the revenue column as currency, or add a total row at the bottom.

# Put your Pandas commands to polish the report with pandas here (example: format revenue as currency):


# Display the polished report
# display(df_report)

---

## 8. Reflection on SQL Reasoning and Debugging (5 points)

Write **1–2 paragraphs** reflecting on your process. Address at least:
- How you broke down your mini-project into smaller SQL questions, or if your mini-project emerged/evolved as you explored the data and wrote individual queries.
- Any bugs, confusing results, or dead ends you hit and how you debugged them (e.g., row-count checks, `LIMIT`, CTE inspection, switching between Adminer and this notebook).
- Any tradeoffs you noticed between expressing logic with CTEs, `GROUP BY`, and window functions (e.g., readability vs. performance, ease of validation) either for this specific assignment or in general based on your experience.


Throughout this project, I approached the analysis by breaking the overall business question into smaller SQL tasks that built on each other. I first focused on join queries to understand how customer activity, rentals, films, and store information were related across the Sakila schema. As I explored the results, the project evolved into a broader analysis of category popularity, customer spending, and store performance. This made the project feel more like a realistic business workflow, where the main question becomes clearer as I continue exploring different aspects of the data overtime.

One of the main challenges I encountered was making sure my joins did not accidentally duplicate rows, especially when working with both rental and payment tables. I used validation techniques such as row-count checks, direct spot checks on individual customers, and intermediate CTEs to confirm that totals were accurate. I also noticed that CTEs improved readability and made debugging easier, while window functions were very useful for calculating percentages and rankings without requiring separate summary queries.This assignment helped strengthen my understanding of how to use SQL effectively and how it can be applied in real business settings.


---

## 9. AI Audit Log (5 points)

This section is about **transparent, professional use of AI tools** (including systems like ChatGPT, GitHub Copilot, Perplexity, or others). You will not lose points for using AI, but you **must** document how you used it.

Answer the prompts below honestly. If you did **not** use any AI tools, you can say so explicitly.


### 9.1 Tools used

- List any AI tools you used while working on this assignment (e.g., ChatGPT, Copilot, Perplexity, Claude, etc.). If none, write "None".
- For each tool, briefly describe what you asked it to help with (e.g., "helped me remember `CREATE INDEX` syntax", "suggested a pattern for a window function").

I used ChatGPT while working on this assignment. I used it to help structure parts of the notebook, improve the wording of written explanations, suggest SQL patterns for joins, CTEs, window functions, and EXPLAIN, and help me think through validation steps for the queries. I also used it to help connect docker adn mysql to jupiter notebook 

### 9.2 Directly used vs. adapted

- Did you paste any AI-generated SQL or code directly into your notebook with minimal changes? If so, point to where (e.g., "Section 5 window query") and **make sure you add a comment in the code cell acknowledging the AI contribution**.
- For code or queries that you **adapted**, briefly describe what you changed and why (e.g., "adjusted table names to match Sakila, simplified to match our validation habits").

> SECTION 1
I directly used because I had trouble connecting sql to my jupiternotebook
> 



### 9.3 Validation and trust

- How did you validate any AI-suggested queries or patterns before trusting them? (e.g., row-count checks, comparing against a simpler query, sanity checking outputs.)
- Are there any parts of your notebook where you are **not fully confident** in the result? If so, describe them, why you're lacking confidence, and what additional checks you would run if you had more time (e.g., "I wasn't sure if the window function was partitioned correctly, I would want to double-check the logic and maybe test it on a smaller dataset if I had more time").

> **I validated AI-assisted queries using row counts, spot checks and grouped totals. I also examined each table and can see that the outputs make since**

- - -

## 10. Creating your GitHub repo and pushing your code (10 points)

Now that you've finished your notebook, it's time to create a GitHub repository and push your code. 

**The goal for this section is to build a repository that someone could clone, spin up the Docker container to access the SQL database, and execute your notebook from top to bottom.**

To achieve this, follow these steps:

1. Create a new private repository on GitHub named `cmse492-hw02-yourMSUNetID` (replace `yourMSUNetID` with your actual MSU NetID).
2. Initialize the repository with a README file.
3. Clone the repository to your local machine.
4. Copy your completed notebook into the cloned repository folder. **Commit the fully-executed version so that someone could see the output _without_ having to run it, but using the Docker set up they should be able to execute it, if needed.**
5. Add the necessary Docker compose file needed to run the Sakila database (you can reuse the one provided in the course materials).
6. Update the README so that it provides an overview of the project, what you accomplished and the skills you're showcasing, and instructions on how to run the notebook and connect to the database (make sure you include information for starting up the Docker containers!).

    - You should be able to share this repository with a friend, co-worker, or future employer and they should be able to understand what you did, why you did it, and how to run your notebook and see your analysis in action.
    
7. Use Git commands to add, commit, and push your changes to GitHub. 

- - -
After you're made sure the GitHub repo is set up correctly, **Submit this notebook to D2L** and **Fill out the MS Form below** to finalize your submission.  
(If the form won't render in your notebook for some reason, you can also access it directly here: https://forms.office.com/r/Ppk0tnVkc8)

In [18]:
from IPython.display import HTML
HTML(
"""
<iframe 
	src="https://forms.office.com/r/Ppk0tnVkc8" 
	width="800" 
	height="800px" 
	frameborder="0" 
	marginheight="0" 
	marginwidth="0">
	Loading...
</iframe>
"""
)

- - -
### Congratulations, you're done!

Submit this assignment by uploading it to the course Desire2Learn web page.  Go to the "Homework" section, find the appropriate submission link, and upload it there. **Make sure your instructors have access to your private GitHub repo by adding them as collaborators!**

See you in class!

---
&#169; Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.